# 11 — Multimodal RAG: Text and Images in a Security Benchmark

## Goal
Explore how a RAG pipeline can combine text with visual evidence. This notebook first retrieves relevant text from the CIS Windows 11 Enterprise Benchmark, then demonstrates a **page-image inspection workflow** for PDF pages that may contain tables, diagrams, or layout details that are hard to recover from extracted text.

Important scope note: the CIS benchmark is primarily a text-and-table PDF. This notebook does not assume that it contains screenshots of your actual Windows endpoint. It renders selected PDF pages to images for inspection and optionally sends a page image to a vision-capable model when the selected provider supports it. The local Ollama/Groq chat configuration below is kept simple; vision support varies by model/provider. If image input is unsupported, the notebook still provides a useful manual image-inspection workflow alongside text RAG.

**Prerequisite:** Run `00b_ingestion_vector_store.ipynb` first. The source PDF must also be available at the path shown below.

## Architecture

```mermaid
flowchart TD
    A["User question"] --> B["Text retrieval from Chroma"]
    A --> C["Select relevant PDF page"]
    C --> D["Render page to image"]
    B --> E["Combine text context"]
    D --> F["Visual inspection / vision model"]
    F --> G["Visual observations"]
    E --> H["Grounded multimodal response"]
    G --> H
    H --> I["Answer + source/page references"]

    classDef input fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px;
    classDef textpath fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:2px;
    classDef imagepath fill:#fef3c7,stroke:#d97706,color:#78350f,stroke-width:2px;
    classDef combine fill:#fce7f3,stroke:#db2777,color:#831843,stroke-width:2px;
    classDef output fill:#ede9fe,stroke:#7c3aed,color:#3b0764,stroke-width:2px;
    class A input;
    class B,E textpath;
    class C,D,F,G imagepath;
    class H combine;
    class I output;
```

## 1. Imports and paths

In [ ]:
from pathlib import Path
import base64
import io
import os
import re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PDF_PATH = (
    PROJECT_ROOT / "data" / "benchmark" /
    "CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1.pdf"
)
VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "chroma_cis_windows11"
COLLECTION_NAME = "cis_windows11_benchmark"

print("Project root:", PROJECT_ROOT)
print("PDF exists:", PDF_PATH.exists())
print("Vector store exists:", VECTOR_STORE_DIR.exists())

## 2. Select a model provider

Choose `ollama` or `groq`, then use `llm` in the notebook. Groq's key is read from `secrets/groq_api_key.txt`. Image input is model-dependent: a text-only model may reject image content. The rest of the notebook remains useful without vision-model support.

In [ ]:
PROVIDER = "ollama"  # "ollama" or "groq"
OLLAMA_MODEL = "gemma4:e2b"
GROQ_MODEL = "llama-3.3-70b-versatile"

if PROVIDER == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print(f"Using Ollama model: {OLLAMA_MODEL}")
elif PROVIDER == "groq":
    from langchain_groq import ChatGroq
    key_path = PROJECT_ROOT / "secrets" / "groq_api_key.txt"
    if not key_path.exists():
        raise FileNotFoundError(
            f"Groq key file not found: {key_path}. Create it and put only your API key in it."
        )
    groq_api_key = key_path.read_text(encoding="utf-8").strip()
    if not groq_api_key:
        raise ValueError(f"Groq key file is empty: {key_path}")
    llm = ChatGroq(model=GROQ_MODEL, api_key=groq_api_key, temperature=0)
    print(f"Using Groq model: {GROQ_MODEL}")
else:
    raise ValueError("PROVIDER must be 'ollama' or 'groq'")

## 3. Connect text retrieval

This keeps the usual text RAG path available so image inspection can be compared with the extracted benchmark text.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
if not VECTOR_STORE_DIR.exists():
    raise FileNotFoundError(
        f"Vector store not found at {VECTOR_STORE_DIR}. "
        "Run notebooks/00b_ingestion_vector_store.ipynb first."
    )

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
    embedding_function=embedding_model,
)
print("Stored record count:", vector_store._collection.count())

## 4. Retrieve text chunks for a question

In [ ]:
from pathlib import Path

def source_label(doc, index: int) -> str:
    meta = doc.metadata or {}
    source = Path(str(meta.get("source", "unknown source"))).name
    page = meta.get("page", "unknown")
    start = meta.get("start_index")
    label = f"[S{index}] {source}, page {page}"
    if start is not None:
        label += f", chunk start {start}"
    return label

def retrieve_text(question: str, k: int = 5):
    return vector_store.similarity_search(question, k=k)

question = "What does the benchmark say about password policy?"
text_docs = retrieve_text(question, k=5)
for i, doc in enumerate(text_docs, start=1):
    print("=" * 88)
    print(source_label(doc, i))
    print(doc.page_content[:900])
    print()

## 5. Render PDF pages to images

Install dependencies in your active notebook environment if needed:

```python
%pip install pymupdf pillow
```

The page index is zero-based in PyMuPDF, while PDF viewers usually display pages starting at 1. Pick a page based on source metadata or inspect a few pages. Rendering is local and does not upload the PDF.

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
from IPython.display import display

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Benchmark PDF not found: {PDF_PATH}")

def render_pdf_page(pdf_path: Path, page_index: int, zoom: float = 1.5) -> Image.Image:
    with fitz.open(pdf_path) as pdf:
        if page_index < 0 or page_index >= len(pdf):
            raise IndexError(f"Page index must be between 0 and {len(pdf)-1}")
        page = pdf.load_page(page_index)
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
        image = Image.open(io.BytesIO(pix.tobytes("png")))
        return image.copy()

# Change this to a relevant zero-based page index after inspecting retrieved metadata.
PAGE_INDEX = 0
page_image = render_pdf_page(PDF_PATH, PAGE_INDEX)
print(f"Rendered PDF page index {PAGE_INDEX} (viewer page {PAGE_INDEX + 1})")
display(page_image)

## 6. Optional: send the rendered page to a vision-capable model

Image support depends on the selected model and integration. The cell below uses a LangChain message containing image data in a common format. Some chat models accept this; others are text-only and will raise an error. If it fails, use the rendered image above for manual inspection or select a vision-capable model/integration.

This cell does not claim that `gemma4:e2b` or `llama-3.3-70b-versatile` necessarily supports images in your installed environment.

In [ ]:
from langchain_core.messages import HumanMessage

def image_to_data_url(image: Image.Image, image_format: str = "PNG") -> str:
    buffer = io.BytesIO()
    image.save(buffer, format=image_format)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    mime = "image/png" if image_format.upper() == "PNG" else "image/jpeg"
    return f"data:{mime};base64,{encoded}"

def ask_about_page_image(image: Image.Image, question: str) -> str:
    data_url = image_to_data_url(image)
    message = HumanMessage(content=[
        {"type": "text", "text": question},
        {"type": "image_url", "image_url": {"url": data_url}},
    ])
    response = llm.invoke([message])
    return response.content if hasattr(response, "content") else str(response)

# Optional experiment. Uncomment after confirming that your chosen model supports image input.
# vision_question = (
#     "Describe the visible structure of this benchmark page. If it contains a table, "
#     "summarize its headings and values. Do not guess text that is unreadable."
# )
# try:
#     vision_observation = ask_about_page_image(page_image, vision_question)
#     print(vision_observation)
# except Exception as exc:
#     print("The selected model/integration may not support image input:", exc)

## 7. Generate a text-grounded answer with page references

This path works even when the selected model is text-only. It answers from retrieved benchmark text and keeps source labels visible.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def build_context(documents, max_chars_per_doc: int = 1600) -> str:
    return "\\n\\n---\\n\\n".join(
        f"{source_label(doc, i)}\\n{doc.page_content.strip()[:max_chars_per_doc]}"
        for i, doc in enumerate(documents, start=1)
    )

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the supplied CIS Windows 11 benchmark text. Cite claims using "
     "source labels such as [S1]. If evidence is insufficient, say so. Do not infer actual "
     "endpoint compliance from benchmark recommendations."),
    ("human", "Question: {question}\\n\\nContext:\\n{context}")
])

context = build_context(text_docs)
response = (answer_prompt | llm).invoke({
    "question": question,
    "context": context or "[No text context retrieved.]",
})
text_answer = response.content if hasattr(response, "content") else str(response)
print(text_answer)

## 8. Compare text and visual observations

If you successfully ran the optional vision cell, store its output in `vision_observation`. Compare it with the text-extracted chunks. The page image can help with layout and tables, while text retrieval is often easier to search across the full document.

Do not combine uncertain visual readings with text evidence as though both were verified. Record page number and inspect the original PDF when details matter.

In [ ]:
# This cell works whether or not the optional vision experiment was run.
if "vision_observation" in globals():
    print("VISUAL OBSERVATION\\n")
    print(vision_observation)
else:
    print(
        "No vision observation is available yet. Render a relevant page and, if supported, "
        "run the optional image-input cell above."
    )

print("\\nTEXT SOURCES\\n")
for i, doc in enumerate(text_docs, start=1):
    print(source_label(doc, i))

## Summary and limitations

- Multimodal RAG can combine text retrieval with visual interpretation of PDF pages.
- Rendering pages helps inspect tables, page layout, and content extraction issues.
- The current example renders pages locally; image understanding is optional because provider/model support varies.
- The page shown is from the benchmark PDF, not a live screenshot of the endpoint.
- Vision models can misread small text and tables. Verify important details against the original PDF.
- For genuine endpoint-state assessment, collect trustworthy machine evidence (for example, approved configuration exports) and retrieve that separately from benchmark recommendations.